# Creating LLM

In [32]:
#reading the verdict from verdict.txt file
with open('verdict.txt', 'r') as file:
    verdict = file.read().strip()
print("The length of the verdict is:", len(verdict))
print(verdict[:100])

The length of the verdict is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [33]:
# Using tiktoken library to tokenize the verdict (BPE tokenization)
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(verdict)
print(encoded[:10])

#decoding the token ids back to text
decoded_ids = tokenizer.decode(encoded)
print(decoded_ids[:100])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [34]:
# Implementing Data Sampling
context_size = 4
x = encoded[:context_size]
y = encoded[1:context_size+1]
print("X",x)
print("Y",y)

X [40, 367, 2885, 1464]
Y [367, 2885, 1464, 1807]


In [35]:
for i in range(1,context_size+1,2):
    context = encoded[:i]
    target = encoded[:i+1]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode(target)}")

I ----> I H
I HAD ----> I HAD always


In [36]:
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, encoded_data, tokenizer, max_length, stride):
        self.input_id = []
        self.target_id = []

        for i in range(0, len(encoded_data)-max_length, stride):
            input_chunk = encoded_data[i:i+max_length]
            target_chunk = encoded_data[i+1:i+max_length+1]
            self.input_id.append(torch.tensor(input_chunk))
            self.target_id.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_id)
    
    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]

In [37]:
# dataset = GPTDataset(encoded, tokenizer, max_length=10, stride=4)
# dataset.__len__()
# dataset.__getitem__(0)

In [38]:
def create_dataloader(txt,batch_size=8, max_length=4, stride=4,shuffle=False,drop_last = True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    data = GPTDataset(tokenizer.encode(txt), tokenizer, max_length, stride)
    dataLoader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataLoader

In [39]:
dataLoader = create_dataloader(verdict,batch_size=8,max_length=4,stride=4,shuffle=False)
# data_itr = iter(dataLoader)
# inputs, targets = next(data_itr)
# print(inputs)
# print(inputs.shape,targets.shape)

#Output :
# tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
# torch.Size([8, 4]) torch.Size([8, 4])

In [40]:
# Embedding the input tokens int the form of 8*4*256
token_encodingLayer = nn.Embedding(50257,256)

# token_encoding = token_encodingLayer(inputs)
# print(token_encoding.shape)

# Output : torch.Size([8, 4, 256])

In [41]:
#Creating positional encodings for the input tokens
pos_embeddingLayer = nn.Embedding(4,256)
pos_encodings = pos_embeddingLayer(torch.arange(4))
print(pos_encodings.shape,pos_encodings)

torch.Size([4, 256]) tensor([[-0.3024,  0.9179,  0.5297,  ...,  0.1233,  0.4187, -0.0241],
        [ 1.8292,  0.1272, -0.4240,  ...,  0.7217, -0.4643,  0.7826],
        [ 0.7492,  0.3377,  1.1739,  ..., -0.5661, -0.4097,  0.3795],
        [-0.3625,  0.8596,  0.0595,  ...,  1.3903,  0.1598,  0.6646]],
       grad_fn=<EmbeddingBackward0>)


In [42]:
# Pocessing each batch
for batch_num, (inputs, targets) in enumerate(dataLoader):
    token_encoding = token_encodingLayer(inputs)
    input_embeddings = token_encoding + pos_encodings
    # print(f"Batch {batch_num}: {token_encoding.shape}")

In [57]:
Wq = nn.Parameter(torch.randn(3,2), requires_grad=True)
x = torch.tensor([0.2, 0., 0.5])    # Shape: [3, 1]
print(x @ Wq)

tensor([ 1.0820, -0.9640], grad_fn=<SqueezeBackward4>)


Creating Attenstion Mechanism